# Phase 4 — Downstream DPO Bias-Propagation (T4 Colab)

Failed-Phase-3 pivot: BOTH comparisons.
- **Arm A**: Policy-RAW vs Policy-HUMAN — does RM-level length bias per se transfer to policy behavior?
- **Arm B**: Policy-RAW vs Policy-REWEIGHT (best reweight seed, r≈+0.289 @ seed0) — does a ~9% RM-level reduction change downstream behavior?

All three policies start from ONE shared SFT checkpoint. Seeds 42 and 0 per arm. Pre-registered decision table, verbatim. Kill-switch: stop if a DPO run is unstable after one LR halving.

**Session-reset safe:** every expensive step (RM training, SFT, DPO, generation) checks Drive FIRST and restores from there before doing any work, and syncs its output back to Drive when done. So if the T4 runtime disconnects mid-run (free tier: ~90 min idle / 12h hard cap), just re-open this notebook, mount Drive, and re-run cells top to bottom — nothing already completed gets redone.

`Runtime → Change runtime type → T4 GPU`, then Run all.


## 1. GPU + pinned deps


In [ ]:
import torch
assert torch.cuda.is_available(), 'Set Runtime -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
%pip install -q 'transformers==5.9.0' 'trl==1.5.0' 'datasets>=2.14.0' 'accelerate>=0.27.0' 'scipy' 'numpy'


## 2. Clone repo + mount Drive


In [ ]:
import os, subprocess
REPO='/content/rlhf-bias-decomp'; BRANCH='phase3-decomposition'
try:
    from google.colab import userdata; token=userdata.get('GITHUB_TOKEN')
except Exception:
    import getpass; token=getpass.getpass('GitHub token: ')
URL=f'https://{token}@github.com/moisheu/rlhf-bias-decomp.git'
if os.path.exists(REPO):
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',REPO,'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',REPO,'pull','origin',BRANCH],check=True)
else:
    subprocess.run(['git','clone','--branch',BRANCH,URL,REPO],check=True)
os.chdir(REPO); print(subprocess.run(['git','log','--oneline','-1'],capture_output=True,text=True).stdout)


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive/rlhf-bias-decomp/phase4'
os.makedirs(DRIVE, exist_ok=True); print('Drive:', DRIVE)


## 2b. Drive restore/sync helpers (session-reset safety)

`restore_dir(local, name)`: if `local` is missing/incomplete but Drive has it, copy it back —
so a fresh VM after a disconnect doesn't retrain something that already finished.
`sync_dir(local, name)`: copy a finished local checkpoint dir to Drive.
Both work on **directories** (checkpoints); plain files (json) are synced inline where used.


In [ ]:
import os, shutil

def _has_weights(d):
    return os.path.exists(os.path.join(d,'model.safetensors')) or os.path.exists(os.path.join(d,'pytorch_model.bin'))

def restore_dir(local_dir, drive_name):
    """If local_dir is missing/incomplete, try restoring it from DRIVE/drive_name."""
    if _has_weights(local_dir):
        return True
    src = f'{DRIVE}/{drive_name}'
    if os.path.isdir(src) and _has_weights(src):
        print(f'  restoring {local_dir} from Drive...')
        shutil.rmtree(local_dir, ignore_errors=True)
        shutil.copytree(src, local_dir)
        return True
    return False

def sync_dir(local_dir, drive_name):
    if _has_weights(local_dir):
        dst = f'{DRIVE}/{drive_name}'
        shutil.copytree(local_dir, dst, dirs_exist_ok=True)
        print(f'  synced {local_dir} -> {dst}')

def restore_file(local_path, drive_name):
    if os.path.exists(local_path):
        return True
    src = f'{DRIVE}/{drive_name}'
    if os.path.exists(src):
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        shutil.copy(src, local_path)
        print(f'  restored {local_path} from Drive')
        return True
    return False

def sync_file(local_path, drive_name):
    if os.path.exists(local_path):
        shutil.copy(local_path, f'{DRIVE}/{drive_name}')

print('Drive restore/sync helpers ready.')


## 3. Prepare relabeler RMs (restore-first, self-contained)

Retrains the two relabeler RMs on this box (or restores them from Drive if a prior session
already trained them), then evals each to confirm its pooled length-r. Raw mixed RM ≈ +0.32;
reweight seed0 ≈ +0.289. Reweight needs the Phase 3 tags+weights, recomputed here (deterministic).


In [ ]:
import os, subprocess, sys
# Phase 3 tags + weights (deterministic; cheap; needed by the reweight RM)
subprocess.run([sys.executable,'-m','experiments.decomposition.build_subset_tags'],check=True)
subprocess.run([sys.executable,'-m','experiments.decomposition.compute_weights'],check=True)

RAW_RM_DIR='results/reward_model_mixed_seed42'
RW_RM_DIR='results/reward_model_reweight_seed0'

# Raw mixed RM (seed 42) — restore from Drive first, else train
if not restore_dir(RAW_RM_DIR, 'rm_mixed_seed42'):
    subprocess.run([sys.executable,'-m','src.train_reward_model'],
                   env={**os.environ,'DATA_MODE':'mixed','TRAIN_SEED':'42'},check=True)
    sync_dir(RAW_RM_DIR, 'rm_mixed_seed42')

# Reweight RM (seed 0) — the Phase-4 corrected relabeler — restore from Drive first, else train
if not restore_dir(RW_RM_DIR, 'rm_reweight_seed0'):
    subprocess.run([sys.executable,'-m','experiments.decomposition.train_phase3'],
                   env={**os.environ,'METHOD':'reweight','TRAIN_SEED':'0','TRAIN_BATCH':'16'},check=True)
    sync_dir(RW_RM_DIR, 'rm_reweight_seed0')

# confirm their length-r (cheap; always re-run)
for d,l in [(RAW_RM_DIR,'raw_relabeler'), (RW_RM_DIR,'reweight_relabeler')]:
    subprocess.run([sys.executable,'-m','experiments.decomposition.eval_length_correlation',
                    '--model-dir',d,'--label',l,'--out','results/phase4_relabeler_rms.json'],check=True)
sync_file('results/phase4_relabeler_rms.json', 'phase4_relabeler_rms.json')
print('Relabeler RMs ready.')


## 4. Build Phase 4 data + relabel (Day 1, restore-first)

Disjoint SFT/DPO 5k slices; relabel the DPO 5k with each RM (+human). Reports agreement & raw-vs-reweight disagreement.


In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable,'-m','experiments.dpo.build_phase4_data'],check=True)

def relabel(labeler, model_dir=None):
    out = f'results/phase4/dpo_labeled_{labeler}.json'
    if restore_file(out, f'dpo_labeled_{labeler}.json'):
        restore_file(f'results/phase4/prefmask_{labeler}.json', f'prefmask_{labeler}.json')
        print(f'SKIP relabel {labeler} (restored from Drive)'); return
    args=['--labeler',labeler] + (['--model-dir',model_dir] if model_dir else [])
    subprocess.run([sys.executable,'-m','experiments.dpo.relabel']+args,check=True)
    sync_file(out, f'dpo_labeled_{labeler}.json')
    sync_file(f'results/phase4/prefmask_{labeler}.json', f'prefmask_{labeler}.json')

relabel('human')
relabel('raw', 'results/reward_model_mixed_seed42')
relabel('reweight', 'results/reward_model_reweight_seed0')


## 5. Shared SFT base (Day 1, restore-first)


In [ ]:
import os, subprocess, sys
SFT_DIR='results/phase4/sft_base'
if not restore_dir(SFT_DIR, 'sft_base'):
    subprocess.run([sys.executable,'-m','experiments.dpo.train_sft'],check=True)
    sync_dir(SFT_DIR, 'sft_base')
print('sft_base ready:', os.path.exists(f'{SFT_DIR}/model.safetensors'))


## 6. DPO runs (Day 2-3) — 3 policies × 2 seeds, restore-first, with kill-switch

Kill-switch: if a run is unstable (NaN/inf loss, exit 2), retry ONCE at half LR; a second
instability stops Phase 4 (do not tune). raw_seed42 runs first (the Day-2 canary).
Each finished DPO checkpoint is synced to Drive immediately so a disconnect never loses it.


In [ ]:
import os, subprocess, sys
def run_dpo(labeler, seed):
    for i, lr in enumerate([5e-6, 2.5e-6]):
        env={**os.environ,'LABELER':labeler,'TRAIN_SEED':str(seed),'DPO_LR':str(lr),'TRAIN_BATCH':'4'}
        rc=subprocess.run([sys.executable,'-u','-m','experiments.dpo.train_dpo'],env=env).returncode
        if rc==0: return True
        if rc==2:
            print(f'  {labeler}_seed{seed} UNSTABLE at lr={lr}' + (' -> halving LR' if i==0 else ' -> KILL-SWITCH'))
            continue
        raise RuntimeError(f'{labeler}_seed{seed} DPO failed rc={rc}')
    return False

ARMS=[('raw',42),('human',42),('reweight',42),('raw',0),('human',0),('reweight',0)]
for lab,s in ARMS:
    tag=f'dpo_{lab}_seed{s}'
    outdir=f'results/{tag}'
    if restore_dir(outdir, tag):
        print(f'SKIP {tag} (restored from Drive)'); continue
    print(f'\n===== DPO {tag} =====', flush=True)
    if not run_dpo(lab,s):
        raise SystemExit(f'KILL-SWITCH: {tag} unstable after one LR halving. Stopping Phase 4 (do not tune).')
    sync_dir(outdir, tag)
print('\nAll DPO runs done.')


## 7. Generate (Day 4) — 7 policies (SFT + 6 DPO), restore-first, cross-scored under both RMs


In [ ]:
import os, subprocess, sys
RAW_RM='results/reward_model_mixed_seed42'; RW_RM='results/reward_model_reweight_seed0'
def gen(policy_dir, label):
    out=f'results/phase4/gen_{label}.json'
    if restore_file(out, f'gen_{label}.json'):
        print(f'SKIP gen_{label} (restored from Drive)'); return
    subprocess.run([sys.executable,'-u','-m','experiments.dpo.generate',
                    '--policy-dir',policy_dir,'--label',label,
                    '--raw-rm',RAW_RM,'--reweight-rm',RW_RM],check=True)
    sync_file(out, f'gen_{label}.json')

gen('results/phase4/sft_base','sft')
for lab in ['raw','human','reweight']:
    for s in [42,0]:
        gen(f'results/dpo_{lab}_seed{s}', f'{lab}_seed{s}')


## 8. Decision table (Day 5) — both arms


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','experiments.dpo.summarize_phase4'])
